# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZonairaTariq/flyrank-AI-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [5]:
# One row represents the daily search performance of one content page for one client. For this assignment, I will use a mid-panel month from the warehouse dataset to explore the data and build features for my lane.

from datasets import load_dataset
import pandas as pd

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance"
)

sample = ds["train"].select(range(1000)).to_pandas()

sample["report_date"] = pd.to_datetime(sample["report_date"])

print("Rows in sample:", len(sample))
print("Date range:")
print(sample["report_date"].min(), "to", sample["report_date"].max())


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Rows in sample: 1000
Date range:
2025-01-27 00:00:00 to 2025-01-30 00:00:00


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [ ]:
### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_sessions

### Label / Proxy
Refresh opportunity score (proxy based on search performance).

### Context
- report_date
- client_hash_id
- content_hash_id

### Excluded
AI traffic columns (ai_chatgpt, ai_gemini, ai_copilot, ai_claude, ai_meta and ai_other) because they are not required for my current refresh scoring task.

In [7]:
sample[[
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions"
]].head()

,report_date,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_sessions
0,2025-01-27,client_9958f0a7ae1df715,content_3b70a18ea133b2bb,30,0,3.833333,0,0
1,2025-01-27,client_9958f0a7ae1df715,content_fe8e8155ce1d47a2,5,0,71.600000,0,0
2,2025-01-27,client_9958f0a7ae1df715,content_b4462a1b90640058,1,0,34.000000,0,0
3,2025-01-27,client_9958f0a7ae1df715,content_c899aef92518c714,6,0,23.333333,0,0
4,2025-01-27,client_9958f0a7ae1df715,content_c7c1d2e68d9d0964,5,0,17.800000,0,0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [16]:
# I verified three things:
#Q1. The grain (one row = one content page for one client on one report date).
#Q2. The row count and date range of my sample.
#Q3. Data availability using the GSC availability field.

#Query 1
print("Total rows:", len(sample))

print("Unique client-content-date combinations:")
print(
    sample[
        ["client_hash_id", "content_hash_id", "report_date"]
    ].drop_duplicates().shape[0]
)
# Query 2

print("Row count:", len(sample))

print("Date range:")
print(sample["report_date"].min())
print(sample["report_date"].max())


# Query 3
available = sample[sample["gsc_data_available"] == True]

print("Rows where gsc_data_available is TRUE:")
print(len(available))

Total rows: 1000
Unique client-content-date combinations:
1000
Row count: 1000
Date range:
2025-01-27 00:00:00
2025-01-30 00:00:00
Rows where gsc_data_available is TRUE:
1000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [17]:
# To understand data leakage, I intentionally included one label-derived column. This can make model performance look unrealistically high because it contains future information. After observing this, I removed the column to keep the experiment fair.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.